In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class Encoder(Model):
  def __init__(self,vocab_size,emb_dim,hidden_dim):
    super(Encoder,self).__init__()
    self.embed = layers.Embedding(vocab_size,emb_dim)
    self.rnn = layers.GRU(hidden_dim,return_state=True,return_sequences=True)

  def call(self,x):
    x = self.embed(x)
    output,hidden = self.rnn(x)
    return hidden

class Decoder(Model):
  def __init__(self,vocab_size,emb_dim,hidden_dim):
    super(Decoder,self).__init__()
    self.embed = layers.Embedding(vocab_size,emb_dim)
    self.rnn = layers.GRU(hidden_dim,return_state=True,return_sequences=True)
    self.fc = layers.Dense(vocab_size)

  def call(self,x,hidden):
    x = self.embed(x)
    output,hidden = self.rnn(x,initial_state=hidden)
    return self.fc(output), hidden

class Seq2Seq(Model):
  def __init__(self,encoder,decoder):
    super(Seq2Seq,self).__init__()
    self.encoder = encoder
    self.decoder = decoder

  def call(self,src,trgt):
    hidden = self.encoder(src)
    output,_ = self.decoder(trgt,hidden)
    return output

# Example usage
vocab_size = 10000  # Example vocab size
emb_dim = 256       # Example embedding dimension
hidden_dim = 512    # Example hidden dimension

encoder = Encoder(vocab_size,emb_dim,hidden_dim)
decoder = Decoder(vocab_size,emb_dim,hidden_dim)
seq2seq = Seq2Seq(encoder,decoder)

# Random input sequences: batch_size=2, sequence_length=5 (for simplicity)
src = tf.random.uniform((2,5),minval=0,maxval=vocab_size,dtype=tf.int32) # Source sequence (batch_size=2, seq_len=5)
tgt = tf.random.uniform((2, 5), minval=0, maxval=vocab_size, dtype=tf.int32)  # Target sequence (batch_size=2, seq_len=5)

# Show input sequences
print("Source Sequence (src):")
print(src)
print("Target Sequence (tgt):")
print(tgt)

# Forward pass through Seq2Seq model
output = seq2seq(src, tgt)

# Show output shape and some sample values
print("\nOutput Shape:")
print(output.shape)  # Expected: (batch_size, seq_len, vocab_size)

# Show the output values for inspection (for a more realistic scenario, you would pass through softmax layer for probabilities)
print("\nOutput (before softmax):")
print(output)

Source Sequence (src):
tf.Tensor(
[[9478 8858 1319 7421 2308]
 [2009  526 1783 6017 6179]], shape=(2, 5), dtype=int32)
Target Sequence (tgt):
tf.Tensor(
[[1989 1037 3689 4045  484]
 [5993 3466 6153 1219 6332]], shape=(2, 5), dtype=int32)

Output Shape:
(2, 5, 10000)

Output (before softmax):
tf.Tensor(
[[[-0.00515893  0.00275286 -0.00265402 ... -0.00128883 -0.00095464
   -0.00121775]
  [-0.00123589 -0.00064797 -0.00227482 ... -0.00018763  0.00240282
   -0.00129848]
  [-0.00057363  0.0017558   0.0009141  ...  0.0009553   0.00334202
   -0.00119411]
  [-0.00391881 -0.00130374  0.00152544 ...  0.0021308   0.00249601
    0.00249778]
  [-0.00446257  0.00073158  0.00034833 ...  0.0021159  -0.00107459
    0.00029517]]

 [[ 0.00013809 -0.00348529 -0.00194819 ...  0.00395953 -0.0001045
   -0.00395374]
  [ 0.00274312 -0.00136338 -0.00081823 ...  0.00183229 -0.00164354
    0.00121146]
  [ 0.00172673 -0.00359656  0.00032053 ... -0.00102954 -0.00266202
    0.00125503]
  [-0.00025753 -0.003348    0.0